#### P4-02
# Logistic Regression Baseline Model
#### M2 - Amri B.S.M
#### IT24102955

## 1. Objective

This notebook develops the baseline Logistic Regression model for hotel booking cancellation prediction.

The model uses the shared-prepared training dataset produced in `07_shared_prep.ipynb`.

The modelling workflow includes:

- a DummyClassifier benchmark
- model-specific one-hot encoding
- numerical feature scaling
- Logistic Regression baseline evaluation
- hyperparameter tuning of `C` and `class_weight`
- 5-fold stratified cross-validation
- ROC-AUC as the tuning metric
- saving the final fitted pipeline using joblib

The test dataset is not used anywhere in this notebook.

In [2]:
%cd /content

/content


In [3]:
!git clone https://github.com/MahdhiS/hotel-cancellation.git

Cloning into 'hotel-cancellation'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 167 (delta 84), reused 79 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 34.04 MiB | 10.66 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [4]:
%cd /content/hotel-cancellation

/content/hotel-cancellation


In [5]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [6]:
!pwd
!ls
!ls src

/content/hotel-cancellation
data  logs  models  notebooks  README.md  requirements.txt  src  template.ipynb
data.py  evaluate.py  features.py  preprocessing.py


In [7]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

import joblib

ROOT = Path("/content/hotel-cancellation")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import split_classifier_data

pd.set_option("display.max_columns", None)

DATA_DIR = ROOT / "data" / "processed" / "shared_prepared"
MODEL_DIR = ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)

Project root: /content/hotel-cancellation
Data directory: /content/hotel-cancellation/data/processed/shared_prepared
Model directory: /content/hotel-cancellation/models


In [8]:
!ls /content/hotel-cancellation/src

data.py  evaluate.py  features.py  preprocessing.py  __pycache__


In [10]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

import joblib

ROOT = Path("/content/hotel-cancellation")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))



pd.set_option("display.max_columns", None)


MODEL_DIR = ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)



In [11]:
!ls src

data.py  evaluate.py  features.py  preprocessing.py  __pycache__


## 2. Load Shared-Prepared Training Data

This notebook uses the final shared-prepared training dataset produced by `07_shared_prep.ipynb`.

Feature engineering, leakage removal, missing-value handling, rare-category grouping, and shared data cleaning have already been completed.

Only the prepared training dataset is loaded for model development. The validation and test datasets are not used during tuning.

In [12]:
train_prepared = pd.read_csv(
    DATA_DIR / "hotel_train_prepared.csv"
)

print("Prepared training shape:", train_prepared.shape)
print("Missing values:", train_prepared.isna().sum().sum())

train_prepared.head()

Prepared training shape: (58192, 29)
Missing values: 0


,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,company,adr,total_nights,total_guests,has_kids,season,arrival_day_of_week
0,2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,Direct,Direct,0,0,0,A,No Deposit,0.0,0.0,75.0,1,1.0,0,Summer,Wednesday
1,3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,Corporate,Corporate,0,0,0,A,No Deposit,Other,0.0,75.0,1,1.0,0,Summer,Wednesday
2,4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,240.0,0.0,98.0,2,2.0,0,Summer,Wednesday
3,6,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0,0,BB,Direct,Direct,0,0,0,C,No Deposit,0.0,0.0,107.0,2,2.0,0,Summer,Wednesday
4,7,Resort Hotel,0,9,2015,July,27,1,0,2,2,0.0,0,FB,Direct,Direct,0,0,0,C,No Deposit,Other,0.0,103.0,2,2.0,0,Summer,Wednesday


In [13]:
print("Rows:", len(train_prepared))
print("Columns:", train_prepared.shape[1])
print("Missing values:", train_prepared.isna().sum().sum())

print("booking_id present:", "booking_id" in train_prepared.columns)
print("country present:", "country" in train_prepared.columns)

Rows: 58192
Columns: 29
Missing values: 0
booking_id present: True
country present: False


In [14]:
X_train, y_train, booking_ids = split_classifier_data(
    train_prepared
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nTarget distribution:")
print(y_train.value_counts())

print("\nTarget proportions:")
print(y_train.value_counts(normalize=True))

X_train shape: (58192, 27)
y_train shape: (58192,)

Target distribution:
is_canceled
0    43451
1    14741
Name: count, dtype: int64

Target proportions:
is_canceled
0    0.746683
1    0.253317
Name: proportion, dtype: float64


In [15]:
assert "booking_id" not in X_train.columns
assert "is_canceled" not in X_train.columns
assert "country" not in X_train.columns

print("Predictor checks passed.")

Predictor checks passed.


In [16]:
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 16
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'total_nights', 'total_guests', 'has_kids']

Categorical features: 11
['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'company', 'season', 'arrival_day_of_week']


In [17]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 3. DummyClassifier Benchmark

A DummyClassifier is used as a no-skill baseline.

This benchmark helps confirm whether Logistic Regression learns meaningful predictive structure beyond simply reflecting the target class distribution.

ROC-AUC is used as the primary comparison metric because model tuning in this project is based on ranking cancellation risk rather than choosing a classification threshold at this stage.

In [18]:
dummy_model = DummyClassifier(
    strategy="prior",
    random_state=42
)

dummy_results = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1
)

dummy_roc_auc = dummy_results["test_roc_auc"].mean()
dummy_pr_auc = dummy_results["test_pr_auc"].mean()

print("Dummy mean ROC-AUC:", round(dummy_roc_auc, 4))
print("Dummy mean PR-AUC:", round(dummy_pr_auc, 4))

Dummy mean ROC-AUC: 0.5
Dummy mean PR-AUC: 0.2533


In [19]:
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

model_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_transformer,
        numeric_features
    ),
    (
        "categorical",
        categorical_transformer,
        categorical_features
    )
])

logistic_pipeline = Pipeline([
    ("preprocessor", model_preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [20]:
baseline_results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1
)

baseline_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "ROC-AUC": baseline_results["test_roc_auc"],
    "PR-AUC": baseline_results["test_pr_auc"]
})

baseline_summary

,Fold,ROC-AUC,PR-AUC
0,1,0.745826,0.530432
1,2,0.749822,0.538142
2,3,0.749523,0.534189
3,4,0.754825,0.545508
4,5,0.754135,0.537944


In [21]:
baseline_roc_auc = baseline_results["test_roc_auc"].mean()
baseline_roc_std = baseline_results["test_roc_auc"].std()
baseline_pr_auc = baseline_results["test_pr_auc"].mean()

print("Baseline mean ROC-AUC:", round(baseline_roc_auc, 4))
print("Baseline ROC-AUC std:", round(baseline_roc_std, 4))
print("Baseline mean PR-AUC:", round(baseline_pr_auc, 4))

Baseline mean ROC-AUC: 0.7508
Baseline ROC-AUC std: 0.0033
Baseline mean PR-AUC: 0.5372


## 4. Hyperparameter Tuning

The Logistic Regression model is tuned using 5-fold stratified cross-validation.

Two hyperparameters are evaluated:

- `C`: controls the strength of regularization. Smaller values apply stronger regularization.
- `class_weight`: controls whether additional importance is assigned to the minority cancellation class.

ROC-AUC is used as the optimization metric because the goal at this stage is to identify the model that best ranks high-risk bookings above low-risk bookings. Classification threshold selection is deferred to a later phase.

The test dataset is not used during tuning.

In [22]:
param_grid = {
    "model__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "model__class_weight": [
        None,
        "balanced"
    ]
}

In [23]:
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(
    X_train,
    y_train
)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['lead_time',
                                                                          'arrival_date_year',
                                                                          'arrival_date_week_number',
                                                                          'arrival_date_day_of_month',
                                                                          'stays_in_weekend_nights',
                                                                          'stays_in_week_nights',
                                                                          'adults',...
                                                                          'arrival_date_month',
                                                                          'meal',
                                                                          'market_segment',
                                                                          'distribution_channel',
                                                                          'reserved_room_type',
                                                                          'deposit_type',
                                                                          'agent',
                                                                          'company',
                                                                          'season',
                                                                          'arrival_day_of_week'])])),
                                       ('model',
                                        LogisticRegression(max_iter=2000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
                         'model__class_weight': [None, 'balanced']},
             return_train_score=True, scoring='roc_auc')

In [24]:
print("Best parameters:")
print(grid_search.best_params_)

print(
    "\nBest mean CV ROC-AUC:",
    round(grid_search.best_score_, 4)
)

Best parameters:
{'model__C': 10.0, 'model__class_weight': 'balanced'}

Best mean CV ROC-AUC: 0.752


In [25]:
tuning_results = pd.DataFrame(
    grid_search.cv_results_
)

tuning_table = tuning_results[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "rank_test_score"
    ]
].copy()

tuning_table = tuning_table.rename(
    columns={
        "param_model__C": "C",
        "param_model__class_weight": "Class Weight",
        "mean_test_score": "Mean CV ROC-AUC",
        "std_test_score": "CV ROC-AUC Std",
        "mean_train_score": "Mean Train ROC-AUC",
        "rank_test_score": "Rank"
    }
)

tuning_table = tuning_table.sort_values(
    "Rank"
).reset_index(drop=True)

tuning_table

,C,Class Weight,Mean CV ROC-AUC,CV ROC-AUC Std,Mean Train ROC-AUC,Rank
0,10.00,balanced,0.751961,0.003309,0.754017,1
1,100.00,balanced,0.751932,0.003322,0.754004,2
2,1.00,balanced,0.751913,0.003333,0.754007,3
3,0.10,balanced,0.751731,0.003406,0.753718,4
4,100.00,None,0.750860,0.003247,0.753035,5
5,10.00,None,0.750838,0.003235,0.753024,6
6,1.00,None,0.750826,0.003306,0.753009,7
7,0.10,None,0.750433,0.003382,0.752532,8
8,0.01,balanced,0.750002,0.003584,0.751702,9
9,0.01,None,0.748254,0.003616,0.750008,10


In [26]:
comparison = pd.DataFrame({
    "Model": [
        "DummyClassifier",
        "Untuned Logistic Regression",
        "Tuned Logistic Regression"
    ],
    "Mean CV ROC-AUC": [
        dummy_roc_auc,
        baseline_roc_auc,
        grid_search.best_score_
    ]
})

comparison

,Model,Mean CV ROC-AUC
0,DummyClassifier,0.500000
1,Untuned Logistic Regression,0.750826
2,Tuned Logistic Regression,0.751961


In [27]:
improvement_over_dummy = (
    grid_search.best_score_ - dummy_roc_auc
)

improvement_over_baseline = (
    grid_search.best_score_ - baseline_roc_auc
)

print(
    "ROC-AUC improvement over Dummy:",
    round(improvement_over_dummy, 4)
)

print(
    "ROC-AUC improvement over untuned Logistic Regression:",
    round(improvement_over_baseline, 4)
)

ROC-AUC improvement over Dummy: 0.252
ROC-AUC improvement over untuned Logistic Regression: 0.0011


## 5. Hyperparameter Tuning Result

The best Logistic Regression configuration was:

- `C = 10.0`
- `class_weight = "balanced"`

The tuned model achieved a mean 5-fold cross-validation ROC-AUC of **0.7520**.

For comparison:

- DummyClassifier ROC-AUC: **0.5000**
- Untuned Logistic Regression ROC-AUC: **0.7508**
- Tuned Logistic Regression ROC-AUC: **0.7520**

Hyperparameter tuning therefore improved ROC-AUC by approximately **0.0011** compared with the untuned Logistic Regression model.

The improvement is small, indicating that the default Logistic Regression configuration was already reasonably stable. However, the balanced class weighting produced the strongest cross-validated ranking performance among the tested configurations and is therefore retained for the final Logistic Regression baseline.

The similarity between mean training ROC-AUC and cross-validation ROC-AUC also indicates no substantial overfitting within the tested parameter grid.

In [28]:
best_logistic_pipeline = grid_search.best_estimator_

final_cv_results = cross_validate(
    best_logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1,
    return_train_score=False
)

final_cv_table = pd.DataFrame({
    "Fold": range(1, 6),
    "ROC-AUC": final_cv_results["test_roc_auc"],
    "PR-AUC": final_cv_results["test_pr_auc"]
})

final_cv_table

,Fold,ROC-AUC,PR-AUC
0,1,0.746793,0.531977
1,2,0.750981,0.539406
2,3,0.750930,0.535618
3,4,0.755911,0.546924
4,5,0.755190,0.538758


In [29]:
final_roc_auc = final_cv_results["test_roc_auc"].mean()
final_roc_std = final_cv_results["test_roc_auc"].std()

final_pr_auc = final_cv_results["test_pr_auc"].mean()
final_pr_std = final_cv_results["test_pr_auc"].std()

print("Final CV ROC-AUC:", round(final_roc_auc, 4))
print("Final ROC-AUC std:", round(final_roc_std, 4))

print("Final CV PR-AUC:", round(final_pr_auc, 4))
print("Final PR-AUC std:", round(final_pr_std, 4))

Final CV ROC-AUC: 0.752
Final ROC-AUC std: 0.0033
Final CV PR-AUC: 0.5385
Final PR-AUC std: 0.005


In [30]:
final_logistic_model = grid_search.best_estimator_

print("Final model:")
print(final_logistic_model)

print("\nBest parameters:")
print(grid_search.best_params_)

Final model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'arrival_date_week_number',
                                                   'arrival_date_day_of_month',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'adults', 'children',
                                                   'babies',
                                                   'is_repeated_guest',
                                                   'previous_cancellations

## 6. Why Logistic Regression Suits the Problem

Logistic Regression is appropriate because hotel booking cancellation is a binary classification problem and the model directly estimates cancellation probability. It provides a simple and interpretable baseline against which more complex classifiers can be compared, while regularization and class weighting allow it to handle many encoded predictors and the observed class imbalance.

In [31]:
results_row = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Configuration": [
        "C=10.0, class_weight=balanced"
    ],
    "CV ROC-AUC": [
        final_roc_auc
    ],
    "CV ROC-AUC Std": [
        final_roc_std
    ],
    "CV PR-AUC": [
        final_pr_auc
    ],
    "Dummy ROC-AUC": [
        dummy_roc_auc
    ]
})

results_row

,Model,Configuration,CV ROC-AUC,CV ROC-AUC Std,CV PR-AUC,Dummy ROC-AUC
0,Logistic Regression,"C=10.0, class_weight=balanced",0.751961,0.003309,0.538536,0.5


## 7. Conclusion

The Logistic Regression baseline was successfully developed using the shared-prepared training data.

A DummyClassifier achieved a ROC-AUC of **0.5000**, while the untuned Logistic Regression achieved **0.7508**.

Grid search over regularization strength and class weighting selected:

- `C = 10.0`
- `class_weight = "balanced"`

with a mean cross-validation ROC-AUC of approximately **0.7520**.

The improvement from tuning was small, but the tuned configuration achieved the strongest ROC-AUC among the tested Logistic Regression settings and is therefore retained as the final Logistic Regression baseline.

The final model-specific pipeline contains one-hot encoding, numerical scaling, and Logistic Regression and has been fitted on the complete training dataset and saved using `joblib`.

The test dataset was not accessed during model development or tuning.